In [ ]:
# 必要なライブラリのimport
import numpy as np
import pandas as pd

#最大表示列数の指定（ここでは50列を指定）
pd.set_option('display.max_columns', 50)

In [ ]:
# google driveへのマウント
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# データ読み込み
simulation_data = pd.read_csv('/content/drive/MyDrive/競馬分析/simulation_data/simulation_20240729.csv')
pay_result = pd.read_csv('/content/drive/MyDrive/競馬分析/data/pay_results.csv')

In [ ]:
# データの確認
simulation_data.head()

,race_id,horse_number,finish_position,target,pred,pred_class
0,202402010901,1,1.0,1,0.721968,1
1,202402010901,2,5.0,0,0.553104,0
2,202402010901,3,2.0,1,0.634559,1
3,202402010901,4,4.0,0,0.235469,0
4,202402010901,5,6.0,0,0.245083,0


In [ ]:
# データの確認
pay_result.head()

,race_id,baken_types,horse_number,refund,popularity
0,202206010101,単勝,15,680,4
1,202206010101,複勝,15br10br4,"210br1,600br170",3br13br1
2,202206010101,枠連,5 - 8,2680,13
3,202206010101,馬連,10 - 15,31040,61
4,202206010101,ワイド,10 - 15br4 - 15br4 - 10,"6,890br660br5,640",58br4br50


### 馬券の買い方一覧(WIN5などその他買い方もあります)  
参考サイト：https://www.jra.go.jp/kouza/beginner/baken/

|  名称  |  説明  |
| ---- | ---- |
|  単勝  |  1着になる馬を当てる馬券  |
|  複勝  |  3着までに入る馬を当てる馬券(出走する馬が7頭以下の場合は、2着までが的中)  |
|  馬連  |  1着と2着になる馬の馬番号の組合せを当てる馬券  |
|  馬単  |  1着と2着になる馬の馬番号を着順通りに当てる馬券  |
|  ワイド  |  3着までに入る2頭の組合せを馬番号で当てる馬券 |
|  3連複  |  1着、2着、3着となる馬の組合せを馬番号で当てる馬券  |
|  3連単  |  1着、2着、3着となる馬の馬番号を着順通りに当てる馬券  |

### 参考：競馬の控除率(仮にすべての馬券を買った場合に帰ってくる金額の割合)  
参考サイト：https://db-keiba.com/return-average/#st-toc-h-3  


|  券種  | 控除率 |  払戻率 |
| ---- | ---- | ---- |
| 単勝 |  20.0% | 80.0% |
| 複勝 | 20.0% | 80.0% |
| 馬連 | 22.5% | 77.5% |
| 馬単 | 25.0% | 75.0% |
| ワイド | 25.0% | 75.0% |
| 三連複 | 25.0%  | 75.0% |
| 三連単 | 27.5%  | 72.5% |

→100%を超えるのが理想ですが、まずはそれぞれの馬券の払戻率を超えることを目標としましょう

## 購入方針の検討
1. 03_モデルの学習.ipynbでそれぞれのレースid毎にpredの確率上位３頭にフラグを立てているので、これをもとに馬券を購入する
2. predの閾値を求め、その閾値以上の馬券を購入する

In [ ]:
class RacePayKinds:
    def __init__(self, dataframe):
        self.return_tables = dataframe

    def _split_columns(self, dataframe, column_name, sep, new_column_prefix):
        return dataframe[column_name].astype(str).str.split(sep, expand=True).add_prefix(new_column_prefix)

    def _prepare_dataframe(self, dataframe, baken_type, horse_sep, refund_sep=None):
        df = dataframe[dataframe["baken_types"] == baken_type][["horse_number", "refund"]]
        wins = self._split_columns(df, "horse_number", horse_sep, "win_")
        returns = self._split_columns(df, "refund", refund_sep or horse_sep, "return_")
        combined_df = pd.concat([wins, returns], axis=1)
        return combined_df.apply(lambda x: pd.to_numeric(x.str.replace(',', ''), errors='coerce')).fillna(0).astype(int)

    def get_race_results(self, race_id, baken_type, horse_sep, refund_sep=None):
        df = self.return_tables[(self.return_tables["race_id"] == race_id) & (self.return_tables["baken_types"] == baken_type)]
        return self._prepare_dataframe(df, baken_type, horse_sep, refund_sep)

    def get_fukusho(self, race_id):
        df = self.get_race_results(race_id, '複勝', 'br')
        # Transform the dataframe to the desired format
        transformed_df = pd.concat([
            pd.DataFrame({'win': df['win_0'], 'return': df['return_0']}),
            pd.DataFrame({'win': df['win_1'], 'return': df['return_1']})
        ], ignore_index=True)
        if 'win_2' in df.columns and 'return_2' in df.columns:
            transformed_df = pd.concat([transformed_df, pd.DataFrame({'win': df['win_2'], 'return': df['return_2']})], ignore_index=True)
        return transformed_df

    def get_tansho(self, race_id):
        return self.get_race_results(race_id, '単勝', None)

    def get_umaren(self, race_id):
        return self.get_race_results(race_id, '馬連', ' - ')

    def get_umatan(self, race_id):
        return self.get_race_results(race_id, '馬単', '→')

    def get_wide(self, race_id):
        wide_df = self.return_tables[(self.return_tables["race_id"] == race_id) & (self.return_tables["baken_types"] == 'ワイド')]
        wins = wide_df["horse_number"].astype(str).str.split('br', expand=True).stack().str.split(' - ', expand=True).add_prefix('win_')
        returns = wide_df["refund"].astype(str).str.split('br', expand=True).stack().str.replace(',', '').rename('return')
        combined_df = pd.concat([wins.reset_index(drop=True), returns.reset_index(drop=True)], axis=1)
        return combined_df.apply(lambda x: pd.to_numeric(x, errors='coerce')).fillna(0).astype(int)

    def get_sanrentan(self, race_id):
        return self.get_race_results(race_id, '三連単', '→')

    def get_sanrenpuku(self, race_id):
        return self.get_race_results(race_id, '三連複', ' - ')

In [ ]:
# データフレームを使ってRacePayKindsクラスをインスタンス化
race_results = RacePayKinds(pay_result)

# 特定のrace_idの複勝の結果を表示
race_results.get_fukusho(202206010101)

,win,return
0,15,210
1,10,1600
2,4,170


In [ ]:
# 特定のrace_idのワイドの結果を表示
race_results.get_wide(202206010101)

,win_0,win_1,return
0,10,15,6890
1,4,15,660
2,4,10,5640


In [ ]:
# ワイドと複勝の回収率を計算
def recovery_rate(race_results, return_tables, bet="wide"):
    race_pay_kinds = RacePayKinds(return_tables)
    targeted_races = {}
    number_of_hits = 0
    amount_of_recovery = 0
    total_rows = 0

    race_id_list = return_tables["race_id"].unique()

    if bet == "wide":
        for race_id in race_id_list:
            pred_numbers = race_results[(race_results["race_id"] == race_id) & (race_results["pred_class"] == 1)]["horse_number"].to_list()
            data = race_pay_kinds.get_wide(race_id)
            if len(pred_numbers) < 3:
              total_rows += 1
            else:
              total_rows += len(data)

            for index, row in data.iterrows():
                if (row["win_0"] in pred_numbers) and (row["win_1"] in pred_numbers):
                    if race_id not in targeted_races:
                        targeted_races[race_id] = []
                    targeted_races[race_id].append({'return': row["return"], 'horses': [row["win_0"], row["win_1"]]})
                    number_of_hits += 1
                    amount_of_recovery += row["return"]

    if bet == "fukusho":
        for race_id in race_id_list:
            pred_numbers = race_results[(race_results["race_id"] == race_id) & (race_results["pred_class"] == 1)]["horse_number"].to_list()
            data = race_pay_kinds.get_fukusho(race_id)
            if len(pred_numbers) < 3:
              total_rows += len(pred_numbers)
            else:
              total_rows += len(data)

            for index, row in data.iterrows():
                if row["win"] in pred_numbers:
                    if race_id not in targeted_races:
                        targeted_races[race_id] = []
                    targeted_races[race_id].append({'return': row["return"], 'horses': [row["win"]]})
                    number_of_hits += 1
                    amount_of_recovery += row["return"]

    hits_rate = number_of_hits / total_rows
    recovery_rate = amount_of_recovery / (total_rows * 100)

    print('的中したレース')
    for race_id, details in targeted_races.items():
        for detail in details:
            print(f'レースID: {race_id}, 的中馬番: {detail["horses"]}, 回収金: {detail["return"]}')
    print('-'* 100)
    print('予測対象レース数', len(race_id_list))
    print('-'* 100)
    print('予測対象馬券数', total_rows)
    print('-'* 100)
    print('的中数', number_of_hits)
    print('-'* 100)
    print('的中率', str(hits_rate * 100) + '%')
    print('-'* 100)
    print('回収金額', amount_of_recovery)
    print('-'* 100)
    print('回収率', str(recovery_rate * 100) + '%')

### 1. の場合
- モデルは3着以内に入る馬を予測するように設計されています。そのため、これに関連する馬券（複勝・ワイド・3連複）での購入結果をシミュレーションしてみます。

In [ ]:
# 予測が1になっているデータのみを取得する
data = simulation_data[simulation_data["pred_class"] == 1]

# 結果データの race_id のリストを取得
result_race_ids = data['race_id'].unique()

# 払戻金データを結果データの race_id のみにフィルタリング
filtered_return_tables = pay_result[pay_result['race_id'].isin(result_race_ids)]

In [ ]:
# ワイドの回収率
recovery_rate(data, filtered_return_tables)

的中したレース
レースID: 202403020301, 的中馬番: [2, 5], 回収金: 140
レースID: 202403020301, 的中馬番: [2, 6], 回収金: 140
レースID: 202403020301, 的中馬番: [5, 6], 回収金: 190
レースID: 202403020302, 的中馬番: [6, 8], 回収金: 360
レースID: 202403020303, 的中馬番: [2, 8], 回収金: 440
レースID: 202403020303, 的中馬番: [1, 8], 回収金: 430
レースID: 202403020303, 的中馬番: [1, 2], 回収金: 330
レースID: 202403020306, 的中馬番: [2, 6], 回収金: 230
レースID: 202403020308, 的中馬番: [1, 8], 回収金: 200
レースID: 202403020309, 的中馬番: [1, 5], 回収金: 240
レースID: 202410030301, 的中馬番: [5, 7], 回収金: 180
レースID: 202410030301, 的中馬番: [5, 6], 回収金: 130
レースID: 202410030301, 的中馬番: [6, 7], 回収金: 290
レースID: 202410030302, 的中馬番: [4, 8], 回収金: 230
レースID: 202410030304, 的中馬番: [1, 2], 回収金: 120
レースID: 202410030305, 的中馬番: [1, 4], 回収金: 240
レースID: 202410030306, 的中馬番: [6, 7], 回収金: 360
レースID: 202410030309, 的中馬番: [2, 3], 回収金: 190
レースID: 202410030309, 的中馬番: [2, 10], 回収金: 360
レースID: 202410030309, 的中馬番: [3, 10], 回収金: 390
レースID: 202410030310, 的中馬番: [1, 4], 回収金: 210
レースID: 202410030311, 的中馬番: [5, 7], 回収金: 310
レースID: 202402010901, 的

In [ ]:
# 複勝の回収率
recovery_rate(data, filtered_return_tables, bet="fukusho")

的中したレース
レースID: 202403020301, 的中馬番: [2], 回収金: 100
レースID: 202403020301, 的中馬番: [5], 回収金: 100
レースID: 202403020301, 的中馬番: [6], 回収金: 110
レースID: 202403020302, 的中馬番: [8], 回収金: 160
レースID: 202403020302, 的中馬番: [6], 回収金: 150
レースID: 202403020303, 的中馬番: [8], 回収金: 180
レースID: 202403020303, 的中馬番: [2], 回収金: 140
レースID: 202403020303, 的中馬番: [1], 回収金: 140
レースID: 202403020305, 的中馬番: [9], 回収金: 200
レースID: 202403020306, 的中馬番: [2], 回収金: 150
レースID: 202403020306, 的中馬番: [6], 回収金: 110
レースID: 202403020307, 的中馬番: [12], 回収金: 130
レースID: 202403020308, 的中馬番: [1], 回収金: 160
レースID: 202403020308, 的中馬番: [8], 回収金: 110
レースID: 202403020309, 的中馬番: [5], 回収金: 160
レースID: 202403020309, 的中馬番: [1], 回収金: 110
レースID: 202403020310, 的中馬番: [3], 回収金: 100
レースID: 202403020312, 的中馬番: [6], 回収金: 210
レースID: 202410030301, 的中馬番: [5], 回収金: 100
レースID: 202410030301, 的中馬番: [7], 回収金: 120
レースID: 202410030301, 的中馬番: [6], 回収金: 100
レースID: 202410030302, 的中馬番: [4], 回収金: 100
レースID: 202410030302, 的中馬番: [8], 回収金: 170
レースID: 202410030304, 的中馬番: [1], 回収金: 120
レースID: 

### 2. の場合
- 閾値を0.7以上、0.8以上などで設けて予測対象馬が１頭の場合は複勝のみ、２頭以上の場合はワイドで購入するようにしてみます。

In [ ]:
# 3頭以内に入る確率が0.7と0.8以上のデータをそれぞれ抽出
data_pred07 = simulation_data[simulation_data['pred'] > 0.7]
data_pred08 = simulation_data[simulation_data['pred'] > 0.8]
print("データ数_0.7以上", len(data_pred07))
print("データ数_0.8以上", len(data_pred08))

データ数_0.7以上 71
データ数_0.8以上 24


In [ ]:
# それぞれのレースごとに予測対象の馬が何頭いるか確認
data_pred07["race_id"].value_counts()

race_id
202402010901    2
202410030304    2
202403020602    2
202403020601    2
202410030501    2
               ..
202410030403    1
202402010902    1
202410030406    1
202410030411    1
202410030608    1
Name: count, Length: 63, dtype: int64

In [ ]:
data_pred08["race_id"].value_counts()

race_id
202403020601    2
202402010907    1
202402011101    1
202410030604    1
202403020612    1
202403020602    1
202402011206    1
202402011205    1
202410030502    1
202410030501    1
202403020505    1
202410030403    1
202403020301    1
202403020410    1
202403020407    1
202403020402    1
202403020401    1
202402011010    1
202410030310    1
202410030302    1
202410030301    1
202403020310    1
202410030606    1
Name: count, dtype: int64

#### predが0.7以上の場合

In [ ]:
# race_idごとにグループ化し、そのサイズを計算
grouped = data_pred07.groupby('race_id').size()

# 行数が1のrace_idを抽出
one_row_race_ids = grouped[grouped == 1].index

# 行数が2のrace_idを抽出
two_row_race_ids = grouped[grouped == 2].index

# 該当する行を抽出
one_row_race_df = data_pred07[data_pred07['race_id'].isin(one_row_race_ids)]
two_row_race_df = data_pred07[data_pred07['race_id'].isin(two_row_race_ids)]

In [ ]:
# 予測が1になっているデータのみを取得する
data = two_row_race_df[two_row_race_df["pred_class"] == 1]

# 結果データの race_id のリストを取得
result_race_ids = data['race_id'].unique()

# 払戻金データを結果データの race_id のみにフィルタリング
filtered_return_tables = pay_result[pay_result['race_id'].isin(result_race_ids)]

# ワイドの回収率
recovery_rate(data, filtered_return_tables)

的中したレース
レースID: 202410030304, 的中馬番: [1, 2], 回収金: 120
レースID: 202402010901, 的中馬番: [1, 6], 回収金: 150
レースID: 202410030405, 的中馬番: [1, 4], 回収金: 110
レースID: 202410030501, 的中馬番: [2, 6], 回収金: 120
レースID: 202403020601, 的中馬番: [1, 2], 回収金: 120
----------------------------------------------------------------------------------------------------
予測対象レース数 8
----------------------------------------------------------------------------------------------------
予測対象馬券数 8
----------------------------------------------------------------------------------------------------
的中数 5
----------------------------------------------------------------------------------------------------
的中率 62.5%
----------------------------------------------------------------------------------------------------
回収金額 620
----------------------------------------------------------------------------------------------------
回収率 77.5%


In [ ]:
# 予測が1になっているデータのみを取得する
data = one_row_race_df[one_row_race_df["pred_class"] == 1]

# 結果データの race_id のリストを取得
result_race_ids = data['race_id'].unique()

# 払戻金データを結果データの race_id のみにフィルタリング
filtered_return_tables = pay_result[pay_result['race_id'].isin(result_race_ids)]

# 複勝の回収率
recovery_rate(data, filtered_return_tables, bet="fukusho")

的中したレース
レースID: 202403020301, 的中馬番: [2], 回収金: 100
レースID: 202403020306, 的中馬番: [6], 回収金: 110
レースID: 202403020308, 的中馬番: [8], 回収金: 110
レースID: 202403020309, 的中馬番: [1], 回収金: 110
レースID: 202403020310, 的中馬番: [3], 回収金: 100
レースID: 202410030301, 的中馬番: [5], 回収金: 100
レースID: 202410030302, 的中馬番: [4], 回収金: 100
レースID: 202410030310, 的中馬番: [1], 回収金: 110
レースID: 202402010903, 的中馬番: [8], 回収金: 110
レースID: 202402010904, 的中馬番: [6], 回収金: 110
レースID: 202402010906, 的中馬番: [3], 回収金: 110
レースID: 202402010907, 的中馬番: [7], 回収金: 110
レースID: 202402010908, 的中馬番: [3], 回収金: 110
レースID: 202402010912, 的中馬番: [6], 回収金: 110
レースID: 202403020401, 的中馬番: [6], 回収金: 100
レースID: 202403020402, 的中馬番: [12], 回収金: 110
レースID: 202403020403, 的中馬番: [15], 回収金: 110
レースID: 202403020410, 的中馬番: [7], 回収金: 110
レースID: 202410030401, 的中馬番: [7], 回収金: 110
レースID: 202410030403, 的中馬番: [2], 回収金: 110
レースID: 202410030406, 的中馬番: [3], 回収金: 130
レースID: 202410030411, 的中馬番: [11], 回収金: 120
レースID: 202402011003, 的中馬番: [11], 回収金: 110
レースID: 202402011005, 的中馬番: [6], 回収金: 110
レースI

#### predが0.8以上の場合

In [ ]:
# race_idごとにグループ化し、そのサイズを計算
grouped = data_pred08.groupby('race_id').size()

# 行数が1のrace_idを抽出
one_row_race_ids = grouped[grouped == 1].index

# 行数が2のrace_idを抽出
two_row_race_ids = grouped[grouped == 2].index

# 該当する行を抽出
one_row_race_df = data_pred08[data_pred08['race_id'].isin(one_row_race_ids)]
two_row_race_df = data_pred08[data_pred08['race_id'].isin(two_row_race_ids)]

In [ ]:
# 予測が1になっているデータのみを取得する
data = two_row_race_df[two_row_race_df["pred_class"] == 1]

# 結果データの race_id のリストを取得
result_race_ids = data['race_id'].unique()

# 払戻金データを結果データの race_id のみにフィルタリング
filtered_return_tables = pay_result[pay_result['race_id'].isin(result_race_ids)]

# ワイドの回収率
recovery_rate(data, filtered_return_tables)

的中したレース
レースID: 202403020601, 的中馬番: [1, 2], 回収金: 120
----------------------------------------------------------------------------------------------------
予測対象レース数 1
----------------------------------------------------------------------------------------------------
予測対象馬券数 1
----------------------------------------------------------------------------------------------------
的中数 1
----------------------------------------------------------------------------------------------------
的中率 100.0%
----------------------------------------------------------------------------------------------------
回収金額 120
----------------------------------------------------------------------------------------------------
回収率 120.0%


In [ ]:
# 予測が1になっているデータのみを取得する
data = one_row_race_df[one_row_race_df["pred_class"] == 1]

# 結果データの race_id のリストを取得
result_race_ids = data['race_id'].unique()

# 払戻金データを結果データの race_id のみにフィルタリング
filtered_return_tables = pay_result[pay_result['race_id'].isin(result_race_ids)]

# 複勝の回収率
recovery_rate(data, filtered_return_tables, bet="fukusho")

的中したレース
レースID: 202403020301, 的中馬番: [2], 回収金: 100
レースID: 202403020310, 的中馬番: [3], 回収金: 100
レースID: 202410030301, 的中馬番: [5], 回収金: 100
レースID: 202410030302, 的中馬番: [4], 回収金: 100
レースID: 202410030310, 的中馬番: [1], 回収金: 110
レースID: 202402010907, 的中馬番: [7], 回収金: 110
レースID: 202403020401, 的中馬番: [6], 回収金: 100
レースID: 202403020402, 的中馬番: [12], 回収金: 110
レースID: 202403020410, 的中馬番: [7], 回収金: 110
レースID: 202410030403, 的中馬番: [2], 回収金: 110
レースID: 202410030501, 的中馬番: [6], 回収金: 110
レースID: 202410030502, 的中馬番: [12], 回収金: 110
レースID: 202402011101, 的中馬番: [5], 回収金: 110
レースID: 202403020602, 的中馬番: [8], 回収金: 110
レースID: 202403020612, 的中馬番: [5], 回収金: 110
レースID: 202410030604, 的中馬番: [12], 回収金: 110
レースID: 202410030606, 的中馬番: [5], 回収金: 110
レースID: 202402011205, 的中馬番: [7], 回収金: 110
レースID: 202402011206, 的中馬番: [12], 回収金: 110
----------------------------------------------------------------------------------------------------
予測対象レース数 22
------------------------------------------------------------------------------------------------

### まとめ
- モデルの予測に基づき全通りで購入した場合、複勝とワイドで控除率に近い回収率を達成できることが確認されました。

- predの確率を0.7以上、0.8以上に絞ることで的中率は向上しましたが、配当金がとても安い馬券のみとなったため、回収率はそれほど高くなりませんでした。

- データとしてはrace_resultテーブルしか利用していないため、過去のレース結果やスピード指数などのデータを組み合わせることで、より精度の高いモデルを作成し、これらの数値を改善する余地があると思われます。